# Carlsen-style Maia bot runner (Lichess: gpt579419)

Runs the bot from https://github.com/eugenecustodio/carlsen-style-maia on this CPU session for about
11.5 hours, then stops taking challenges, finishes games in progress and exits, before Kaggle's
12-hour limit. Relaunch: `kaggle kernels push -p kaggle/` or Save Version → Save & Run All.

The Lichess token comes from the private dataset `jakenorton23/carlsen-bot-token` and is never printed.

In [ ]:
import glob, os, subprocess, sys, time, io, tarfile, urllib.request
import yaml

REPO = 'https://github.com/eugenecustodio/carlsen-style-maia.git'
LICHESS_BOT_COMMIT = 'df7e730de58cc3ef2f1415a0dc2eeda842d39167'
BOT_DIR, LB_DIR = '/tmp/bot', '/tmp/lichess-bot'          # /tmp, not /kaggle/working: keep the model out of the outputs
RUN_HOURS = 11.5

subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, BOT_DIR], check=True)
url = f'https://github.com/lichess-bot-devs/lichess-bot/archive/{LICHESS_BOT_COMMIT}.tar.gz'
tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen(url).read())).extractall('/tmp/lb')
os.rename(glob.glob('/tmp/lb/lichess-bot-*')[0], LB_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{BOT_DIR}/requirements.txt',
                '-r', f'{LB_DIR}/requirements.txt'], check=True)
subprocess.run([sys.executable, f'{BOT_DIR}/make_config.py', f'{LB_DIR}/config.yml.default', f'{LB_DIR}/config.yml',
                '--engine-dir', f'{BOT_DIR}/engine', '--fast'], check=True)
cfg = yaml.safe_load(open(f'{LB_DIR}/config.yml'))
cfg['engine']['interpreter'] = sys.executable            # the engine runs with this kernel's Python
yaml.safe_dump(cfg, open(f'{LB_DIR}/config.yml', 'w'), sort_keys=False)
print('commit:', subprocess.run(['git', '-C', BOT_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
print('CPU cores:', os.cpu_count())

In [ ]:
token_file = glob.glob('/kaggle/input/**/token.txt', recursive=True)
assert token_file, 'attach the private dataset jakenorton23/carlsen-bot-token'
env = dict(os.environ, LICHESS_BOT_TOKEN=open(token_file[0]).read().strip(), LICHESS_BOT_NAME='gpt579419',
           LICHESS_BOT_DIR=LB_DIR, KEEPALIVE='0', PORT='10000', CARLSEN_THREADS='2',
           BOT_RUN_SECONDS=str(int(RUN_HOURS * 3600)), BOT_GRACE_SECONDS='900')
t0 = time.time()
print(f'starting the bot for {RUN_HOURS} h (until about {time.strftime("%H:%M UTC", time.gmtime(t0 + RUN_HOURS * 3600))})', flush=True)
p = subprocess.Popen([sys.executable, 'server.py'], cwd=BOT_DIR, env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
print(f'bot stopped with code {p.wait()} after {(time.time() - t0) / 3600:.2f} h', flush=True)